<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Insumos de la matriz insumo-producto y del valor agregado sectorial

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Multiplicadores de Leontief

Carga la matriz inversa de Leontief del cuadro 8 de la matriz insumo-producto de 2019, que
traduce un choque de demanda final en producción por rama de actividad.

In [3]:
(temp / 'pib').mkdir(parents=True, exist_ok=True)

df_leontief = pd.read_excel(raw / 'pib' / 'anex-MIP-2019.xlsx', sheet_name='Cuadro 8', header=10).dropna().drop(columns=['Unnamed: 1']).rename(columns={'Unnamed: 0':'CIIU'})
df_leontief["CIIU"] = df_leontief["CIIU"].astype(str)
df_leontief.set_index('CIIU', inplace=True)
df_leontief.to_parquet(temp / 'pib' / 'multiplicadores_leontief.parquet')

### Correspondencia entre la CIIU y las ramas de la matriz insumo-producto

Completa la jerarquía de la clasificación industrial y la cruza con los nombres de rama del
cuadro 8 para obtener la equivalencia entre clase CIIU y rama de la matriz.

**Nota de auditoría:** el cruce por descripción solo resuelve una parte de las filas. El archivo
`CIIU-complementado.xlsx` que consume el paso siguiente tiene el resto completado a mano en
Excel, y es el que vive en `data/raw`. Esta celda se conserva porque documenta cómo se sembró
ese archivo, pero su salida va a `data/temp` y la cadena no la consume. En el proyecto original
esta celda nunca llegó a ejecutarse.

In [4]:
df_ciiu_mio = pd.read_excel(raw / 'ciiu' / 'CIIU-básico.xlsx', sheet_name='Hoja1')
df_ciiu_mio[['Sección','División', 'Grupo']] = df_ciiu_mio[['Sección','División', 'Grupo']].fillna(method="ffill")
df_ciiu_mio['Grupo'] = df_ciiu_mio['Grupo'].fillna(0).astype(int).astype(str).str.zfill(3)
df_ciiu_mio['Clase'] = df_ciiu_mio['Clase'].fillna(0).astype(int).astype(str).str.zfill(4)

df_ciiu__mio_nombres = pd.read_excel(raw / 'pib' / 'anex-MIP-2019.xlsx', sheet_name='Cuadro 8', header=10)[['Unnamed: 0',  'Unnamed: 1']].dropna()
df_ciiu__mio_nombres.rename(columns={'Unnamed: 0': 'CIIU MIO', 'Unnamed: 1': 'Descripción'}, inplace=True)

# CAMBIO DE MIGRACION: el original escribia CIIU-complementado.xlsx dentro de data/raw,
# es decir, mutaba el archivo crudo en cada corrida. Aqui la salida va a data/temp y el
# paso siguiente la lee desde alli, con lo que data/raw queda inmutable.
# Ver tools/markdown/AUDITORIA_MIGRACION.md
(temp / "pib").mkdir(parents=True, exist_ok=True)
df_ciiu_mio.merge(df_ciiu__mio_nombres, on='Descripción', how='left').to_excel(temp / "pib" / "CIIU-complementado.xlsx", index=False)

### Equivalencia final por clase

Conserva una fila por clase CIIU con su rama correspondiente, a partir del archivo completado a
mano. Las filas sin clase asignada se descartan: de 715 quedan 499.

In [5]:
# La cadena consume la version completada a mano que vive en data/raw: el cruce por
# 'Descripción' de la celda anterior solo resuelve 69 de las 715 filas, y alguien
# completo el resto en Excel. Verificado: leyendo de data/raw el resultado es identico
# al original, 499 filas; con el regenerado serian 715.
(temp / 'pib').mkdir(parents=True, exist_ok=True)

df_ciiu_mio_final = pd.read_excel(raw / 'ciiu' / 'CIIU-complementado.xlsx').dropna(subset=['Clase'])
df_ciiu_mio_final['Clase'] = df_ciiu_mio_final['Clase'].astype(int).astype(str).str.zfill(4)
df_ciiu_mio_final['CIIU'] = df_ciiu_mio_final['Clase']
df_ciiu_mio_final = df_ciiu_mio_final[['CIIU', 'CIIU MIO']]
df_ciiu_mio_final['CIIU MIO'] = df_ciiu_mio_final['CIIU MIO'].astype(str)
df_ciiu_mio_final.set_index('CIIU MIO', inplace=True)
df_ciiu_mio_final.to_parquet(temp / "pib" / "df_ciiu_mio.parquet")

### Valor agregado por rama

Extrae del cuadro 32 de las cuentas nacionales anuales las filas de valor agregado y las
transpone a una fila por rama.

In [6]:
(temp / 'pib').mkdir(parents=True, exist_ok=True)

df_va = (
    pd.read_excel(raw / "pib" / "anex-CuentasNalANuales-OfertaUtilizacionPreciosCorrientes-2023p.xlsx", sheet_name='Cuadro 32', header=10)
    .loc[399:401,:]
    [['Unnamed: 1','A0101-01\nAgricultura y actividades de servicios conexas ','A0101-02\nCultivo permanente de café','A0102\nGanadería, caza y actividades de servicios conexas ','A02\nSilvicultura y extracción de madera','A03\nPesca y acuicultura','017\nExtracción de carbón de piedra y lignito','018 + 021\nExtracción de petróleo crudo y gas natural y actividades de apoyo para la extracción de petróleo y de gas natural','019\nExtracción de minerales metalíferos','020\nExtracción de otras minas y canteras','022\nActividades de apoyo para otras actividades de explotación de minas y canteras','023 + 024 + 025\nProcesamiento y conservación de carne y productos cárnicos de bovinos, bufalinos, porcinos y otras carnes n.c.p.; procesamiento y conservación de carne y productos cárnicos de aves de corral y procesamiento y conservación de pescados, crustáceos y moluscos','026\nElaboración de aceites y grasas de origen vegetal y animal','027\nElaboración de productos lácteos','028 + 032 + 035\nElaboración de productos de molinería, almidones y productos derivados del almidón; elaboración de productos de panadería; elaboración de macarrones, fideos, alcuzcuz, y productos farináceos similares y elaboración de alimentos preparados para animales','029\nElaboración de productos de café','030 + 031\nElaboración de azúcar y elaboración de panela','033\nElaboración de cacao, chocolate y productos de confitería','034\nProcesamiento y conservación de frutas, legumbres, hortalizas y tubérculos; elaboración de otros productos alimenticios (platos preparados y conservados mediante enlatado o congelado, elaboración de sopas y caldos en estado sólidos, polvo o instantáneas entre otros)','036\nElaboración de bebidas (incluido el hielo) y elaboración de productos de tabaco','037 + 038\nPreparación, hilatura, tejeduría y acabado de productos textiles; fabricación de otros productos textiles, Confección de prendas de vestir','039\nCurtido y recurtido de cueros; fabricación de calzado; fabricación de artículos de viaje, maletas, bolsos de mano y artículos similares, y fabricación de artículos de talabartería y guarnicionería; adobo y teñido de pieles','040\nTransformación de la madera y fabricación de productos de madera y de corcho, excepto muebles; fabricación de artículos de cestería y espartería','041\nFabricación de papel, cartón y productos de papel y de cartón','042\nActividades de impresión; producción de copias a partir de grabaciones originales (Copia a partir de un original en CD, DVD, Bluray)','043 + 044\nCoquización, fabricación de productos de la refinación del petróleo y actividades de mezcla de combustibles','045 + 046 + 047\nFabricación de sustancias químicas básicas, abonos y compuestos inorgánicos nitrogenados, plásticos y caucho sintético en formas primarias; fabricación de otros productos químicos; fabricación de fibras sintéticas y artificiales; fabricación de productos farmacéuticos, sustancias químicas medicinales y productos botánicos de uso farmacéutico','048\nFabricación de productos de caucho y de plástico','049\nFabricación de otros productos minerales no metálicos','050 + 051\nFabricación de productos metalúrgicos básicos; fabricación de productos elaborados de metal, excepto maquinaria y equipo','052\nFabricación de aparatos y equipo eléctrico; fabricación de productos informáticos, electrónicos y ópticos','053 + 057\nFabricación de maquinaria y equipo n.c.p.; instalación, mantenimiento y reparación especializado de maquinaria y equipo','054\nFabricación de vehículos automotores, remolques y semirremolques; fabricación de otros tipos de equipo de transporte','055\nFabricación de muebles, colchones y somieres','056\nOtras industrias manufactureras','058 + 059 + 060\nGeneración de energía eléctrica; transmisión de energía eléctrica y distribución y comercialización de energía eléctrica','061\nProducción de gas; distribución de combustibles gaseosos por tuberías; suministro de vapor y aire acondicionado','062\nCaptación, tratamiento y distribución de agua','063+064+066\nEvacuación y tratamiento de aguas residuales; recolección, tratamiento y disposición de desechos y actividades de saneamiento ambiental y otros servicios de gestión de desechos','065\nRecuperación de materiales (reciclaje)','067\nConstrucción de edificaciones residenciales y no residenciales','068\nConstrucción de carreteras y vías de ferrocarril, de proyectos de servicio público y de otras obras de ingeniería civil','069\nActividades especializadas para la construcción de edificaciones y obras de ingeniería civil (Alquiler de maquinaría y equipo de construcción con operadores)','070\nComercio al por mayor y en comisión o por contrata; comercio al por menor (incluso el comercio al por menor de combustibles); comercio de vehículos automotores y motocicletas, sus partes, piezas y accesorios','071\nMantenimiento y reparación de vehículos automotores y motocicletas','072 + 074\nTransporte terrestre y transporte por tuberías','073\nTransporte acuático','075\nTransporte aéreo','076\nAlmacenamiento y actividades complementarias al transporte','077\nActividades de correo y de servicios de mensajería','I\nAlojamiento y servicios de comida','J\nInformación y comunicaciones','K\nActividades financieras y de seguros','089\nActividades inmobiliarias','M\nActividades profesionales, científicas y técnicas','N\nActividades de servicios administrativos y de apoyo','O\nAdministración pública y defensa; planes de seguridad social de afiliación obligatoria','100\nEducación de mercado','101\nEducación de no mercado','Q\nActividades de atención de la salud humana y de servicios sociales','R + S\nActividades artísticas, de entretenimiento y recreación y otras actividades de servicios','109\nActividades de los hogares individuales en calidad de empleadores']]
    .set_index('Unnamed: 1')
    .T
)
df_va.to_parquet(temp / "pib" / "df_va.parquet", index=True)
df_va.to_excel(temp / "pib" / "df_va.xlsx", index=True)